In [1]:
import os
import re
import time
import pandas as pd


In [16]:
INPUT_DIR = "/home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election"

def find_all_files_recursively(directory, extension=".csv.gz"):
    files = []
    for root, _, filenames in os.walk(directory):
        for filename in filenames:
            if filename.endswith(extension):
                files.append(os.path.join(root, filename))
    return files


def inspect_location_data(file_path):
    try:
        header = pd.read_csv(
            file_path,
            compression="gzip",
            nrows=0
        ).columns

        has_location = "location" in header
        has_lang = "lang" in header

        if not has_location or not has_lang:
            return {
                "file": file_path,
                "has_location_column": False,
                "total_rows": 0,
                "location_present": 0
            }

        df = pd.read_csv(
            file_path,
            compression="gzip",
            usecols=["location", "lang"],
            dtype=str,
            low_memory=False
        )

        df = df[df["lang"] == "en"]

        return {
            "file": file_path,
            "has_location_column": True,
            "total_rows": len(df),
            "location_present": df["location"].notna().sum(),
            "location_missing": df["location"].isna().sum()
        }

    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None



In [17]:
print(INPUT_DIR)

/home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election


In [18]:
files = find_all_files_recursively(INPUT_DIR)
print(f"Found {len(files)} files")

summaries = []
for i, file in enumerate(files):
    result = inspect_location_data(file)
    if result:
        summaries.append(result)

    if i % 50 == 0:
        print(f"Processed {i}/{len(files)} files")


Found 881 files
Processed 0/881 files
Processed 50/881 files
Processed 100/881 files
Processed 150/881 files
Processed 200/881 files
Processed 250/881 files
Processed 300/881 files
Processed 350/881 files
Processed 400/881 files
Processed 450/881 files
Processed 500/881 files
Processed 550/881 files
Processed 600/881 files
Processed 650/881 files
Processed 700/881 files
Processed 750/881 files
Processed 800/881 files
Processed 850/881 files


In [19]:
summary_df = pd.DataFrame(summaries)

print(summary_df[["total_rows", "location_present", "location_missing"]].describe())

# Overall availability
total_rows = summary_df["total_rows"].sum()
total_locations = summary_df["location_present"].sum()

print(f"\nOverall rows: {total_rows}")
print(f"Rows with location data: {total_locations}")
print(f"Location availability: {total_locations / total_rows:.2%}")


         total_rows  location_present  location_missing
count    881.000000        881.000000        834.000000
mean   39533.811578       1734.544835      39929.441247
std    12287.100187       7365.444215      10820.035593
min        0.000000          0.000000          0.000000
25%    41929.000000          0.000000      41686.250000
50%    43598.000000          0.000000      43542.000000
75%    44813.000000          0.000000      44777.250000
max    47160.000000      45720.000000      47160.000000

Overall rows: 34829288
Rows with location data: 1528134
Location availability: 4.39%


In [22]:
summary_df[summary_df['location_present'] > 0]

,file,has_location_column,total_rows,location_present,location_missing
4,/home/tsaddi01/msc-project-source-code-files-2...,True,43342,12928,30414.0
10,/home/tsaddi01/msc-project-source-code-files-2...,True,45899,3347,42552.0
87,/home/tsaddi01/msc-project-source-code-files-2...,True,44354,513,43841.0
173,/home/tsaddi01/msc-project-source-code-files-2...,True,45595,5813,39782.0
176,/home/tsaddi01/msc-project-source-code-files-2...,True,43001,25765,17236.0
...,...,...,...,...,...
753,/home/tsaddi01/msc-project-source-code-files-2...,True,42088,25858,16230.0
774,/home/tsaddi01/msc-project-source-code-files-2...,True,45588,13200,32388.0
792,/home/tsaddi01/msc-project-source-code-files-2...,True,43727,21277,22450.0
804,/home/tsaddi01/msc-project-source-code-files-2...,True,40991,32401,8590.0
